# Module 5: LangSmith Engine — Close the Loop Automatically

> *"Your proactive agent engineer."* — LangSmith

> Part of the **Modular Workshops** series. Standalone, ~30 min. Runs as an alternative to Module 4, or a faster lap right after it.

In Module 4 you built the improvement loop **by hand**. **Engine automates the whole thing.** It watches your deployed agent's traces and runs five stages on its own:

1. **Detect** — find a *recurring* failure across many traces
2. **Diagnose** — trace the root cause into your connected GitHub source
3. **Fix** — open a pull request (code or prompt)
4. **Guard** — deploy an online evaluator so the issue can't silently regress
5. **Reopen** — if it resurfaces after being resolved, reopen it

To give Engine something to find, we run the deployed `deep_agent` (Module 3) through an **assistant** called `engine-demo` that swaps in a subtly broken search tool. Engine scans in the background (~20 min first pass, then on a cron), so a presenter primes it ahead. The rest of the module follows what Engine works from — **traces**, **context**, **evaluators** — then recaps.

<img src="../images/engine_issue_view.png" style="width: auto; max-height: 360px; border-radius: 8px;">

## Setup

Deployed runs trace to the deployment's own project — **`modular-workshops-deep-agent`** — so that's the project Engine analyzes and the one we point traces, datasets, and evaluators at. The cell below resolves the deployment URL and upserts the `engine-demo` assistant (the broken-search config lives in `utils/engine.py`).

In [ ]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(dotenv_path=project_root / ".env", override=True)

from langsmith import Client
from langgraph_sdk import get_sync_client
from utils.engine import deployment_url, upsert_engine_assistant

project_name = "modular-workshops-deep-agent"
client = Client()
sdk = get_sync_client(url=deployment_url(project_name))
assistant_id = upsert_engine_assistant(sdk)
print("engine-demo assistant:", assistant_id)

### Seed a few traces

Ask `engine-demo` a handful of questions. Because it uses the broken search tool, every answer comes back vague and ungrounded — the recurring failure Engine detects. A couple of traces is enough.

In [ ]:
research_prompts = [
    "What is LangGraph used for?",
    "What is LangSmith for?",
    "What is the difference between LangChain and LangGraph?",
]

for q in research_prompts:
    sdk.runs.create(None, assistant_id, input={"messages": [{"role": "user", "content": q}]})
    print("seeded:", q)

print(f"\nSeeded {len(research_prompts)} traces into project: {project_name}")

### The issue Engine filed

For each recurring failure, Engine opens an **issue** with a title, severity, and root-cause description. Browse them in the **Engine** tab, or list them from the CLI:

In [ ]:
!langsmith project issues list --project "{project_name}" --format pretty

## 1. Traces

Engine starts from **traces**. It clusters recurring failures and links the exact runs behind each issue — open **Linked Traces** to inspect them. Here those runs show `easy_search` returning titles and URLs but **no page content**, so the agent answers without grounding. Peek at a few search outputs to see the gap:

In [ ]:
print("Inspect traces:", client.read_project(project_name=project_name).url, "\n")

for r in client.list_runs(project_name=project_name, filter='eq(name, "easy_search")', limit=3):
    print("-", str(r.outputs)[:160])

## 2. Context

A trace says *what* broke; **context** says *why*. Engine reads everything you connect — your **GitHub source** plus the agent's runtime context (prompts, skills, memory).

This assistant loads its guidance from the Context Hub repo `engine-demo-context`. Pull it straight from the repo the agent reads at runtime:

In [ ]:
from deepagents.backends.context_hub import ContextHubBackend

hub = ContextHubBackend("engine-demo-context")
print(hub.read("AGENTS.md").file_data["content"])

That guidance is deliberately weak — *"lead with your own knowledge," "one search is enough," "don't cite sources"* — which, paired with the broken tool, pushes the agent toward confident, ungrounded answers.

Reading both the source and this context, Engine pins the root cause and writes a **fix**: it shows the diff with an explanation and can open a **PR** — code *or* prompt. Review it like a teammate's.

## 3. Evaluators

A fix closes today's gap; an **evaluator** keeps it from coming back. Engine does two things here, both in one click in the UI:

- **Dataset** — it turns the failing traces into ground-truth examples and appends them to a dataset, so the fix is backed by data you can re-run.
- **Online evaluator** — it deploys a check that scores every new trace and reopens the issue if the regression returns (the same online evaluator you built by hand in Module 4).

The cell below creates the seed dataset Engine appends to.

In [ ]:
dataset_name = "engine-demo-evals"

if not client.has_dataset(dataset_name=dataset_name):
    ds = client.create_dataset(dataset_name)
    client.create_examples(dataset_id=ds.id, examples=[
        {"inputs": {"question": "What is LangGraph used for?"},
         "outputs": {"reference_answer": "Grounded explanation, citing sources, that LangGraph orchestrates stateful agent workflows."}},
        {"inputs": {"question": "What is LangSmith for?"},
         "outputs": {"reference_answer": "Grounded explanation, citing sources, of tracing, evals, and monitoring for LLM apps."}},
    ])
else:
    ds = client.read_dataset(dataset_name=dataset_name)

print("Dataset ready — Engine appends ground-truth examples here:", ds.url)

## Recap

Same loop you built by hand in Module 4 — now detected, diagnosed, fixed, and guarded automatically.

| Engine stage | By hand in Module 4 | Engine automates |
|---|---|---|
| **Detect** | `list_runs` + filters | Finds recurring failures on its own |
| **Diagnose** | Read traces, guessed | Traces the cause into GitHub source |
| **Fix** | Edited code yourself | Opens a PR (code or prompt) |
| **Guard** | `create_run_rule(...)` | Deploys an evaluator in one click |
| **Re-check** | Re-ran evals manually | Rescans every ~6h, auto-reopens regressions |

**Close the loop:** merge Engine's PR → redeploy (`langgraph deploy`) → re-run the seed cell. Answers come back grounded and Engine resolves the issue on its next rescan. Revert the fix to demo again. Engine also runs on its own — rescanning every ~6h, emitting webhooks, metered in LCUs (1 LCU = $1.50).

**Docs:** [LangSmith Engine](https://docs.langchain.com/langsmith/engine) · [Assistants](https://docs.langchain.com/langsmith/assistants) · [Engine webhooks](https://docs.langchain.com/langsmith/engine-webhooks)

In [ ]:
import json, subprocess

raw = subprocess.run(
    ["langsmith", "project", "issues", "list", "--project", project_name, "--format", "json"],
    capture_output=True, text=True,
).stdout
issues = json.loads(raw or "[]")

print(f"Engine filed {len(issues)} issue(s) for {project_name}:\n")
for i in issues:
    fix = f"PR #{i['fix_pr_number']}" if i.get("fix_pr_number") else f"draft branch {i['fix_branch']}"
    print(f"[sev {i['severity']}] {i['status']:6} · {i['name']}")
    print(f"   tags: {', '.join(i['tags']) or '—'} · fix: {fix}\n")